# 04 — Feature Extraction

Extract and save feature datasets from PSG recordings for machine learning.

**Notebook Objectives**

- Build epoch-level datasets from EDF + hypnogram pairs.
- Extract time- and frequency-domain features per epoch.
- Persist generated feature datasets for downstream training.

**Contents**

1. Setup and imports
2. Load a sample sleep session
3. Build raw epoch dataset
4. Extract features with `feature_extractor`
5. Save feature datasets to disk

**Inputs**

- EDF + hypnogram pairs from data/dataset/physionet-sleep-data

**Outputs**

- Feature dataset files saved to the project's datasets directory

In [ ]:
# --- configure the Python path to allow imports from the src under project root ---
import sys
from pathlib import Path

# add src folder parent (the Project root) to sys.path
sys.path.insert(0, str(Path.cwd().parent))
print("Added to sys.path:", str(Path.cwd().parent))

In [ ]:
# --- Local imports ---
from src import constants as const
from src import folders as fldr
from src import sleep_session as ss
from src import feature_extractor as fe
from src import dataset_io as dsio

In [ ]:
# --- Retrieve sample sleep session file pair (EDF and hypnogram)  ---
subject_id, psg_file, hypno_file = fldr.get_edf_file_pairs()[0] # Select first for demonstation 
print(f"Subject ID: {subject_id}")
print(f"PSG file: {psg_file}")
print(f"Hypnogram file: {hypno_file}")


In [ ]:
# --- create a SleepSession object for feature extraction ---
session = ss.SleepSession(edf_path=psg_file, hypno_path=hypno_file)

# --- build dataset (single channel + hypnogram) for ML training ---
# Extract raw sleep dataset from a specific EEG channel with 30-second epochs
ds = session.get_sleep_dataset(
    channel=const.CHANNEL_SELECTION,  # Select the Fpz-Cz electrode channel
    epoch_len_s=const.EPOCH_LENGTH_SECONDS,  # Use 30-second epochs as per standard sleep analysis
    filter_cfg=ss.FilterConfig(
        enabled=False,  # Disable filtering for this example (use raw signal)
    ),
    remap_cfg=ss.MappingConfig(
        enabled=True,  # Enable label remapping (use original sleep stages)
        remap=const.SLEEP_STAGE_MAP,  # Mapping of original sleep stages to new labels (if needed)
    ),
)


In [ ]:
# --- Extract features and print summary ---    
# Build feature-extracted dataset with time and frequency domain features
dataset: fe.Dataset = fe.build_dataset(
    epochs=ds.epochs,  # Raw EEG epochs extracted from the sleep session
    labels=ds.unique_labels,  # Sleep stage labels (Wake, N1, N2, N3, REM)
    sampling_freq_hz=ds.sampling_rate,  # Sampling frequency of the EEG signal
    n_bins=25,  # Number of frequency bins for FFT-based feature extraction
)
# Print a formatted summary of the feature-extracted dataset
print(fe.format_feature_dataset_summary(dataset))

In [ ]:
# --- Write the feature dataset to disk for later use in ML training ---

subjects = fldr.get_edf_file_pairs()
counter = 0

print("-" * 50)
print (f"Extracting features for {len(subjects)} subjects to generate dataset for machine learning...")
print("-" * 50)
for subject_id, psg_file, hypno_file in subjects:
    # Create a SleepSession object for the current subject
    session = ss.SleepSession(edf_path=psg_file, hypno_path=hypno_file)
    
    # Build the raw sleep dataset (epochs and labels) for the current subject
    ds = session.get_sleep_dataset(
        channel=const.CHANNEL_SELECTION,
        epoch_len_s=const.EPOCH_LENGTH_SECONDS,
        filter_cfg=ss.FilterConfig(enabled=False),
        remap_cfg=ss.MappingConfig(enabled=True, remap=const.SLEEP_STAGE_MAP),
    )
    
    # Extract features from the raw sleep dataset for the current subject
    dataset: fe.Dataset = fe.build_dataset(
        epochs=ds.epochs,
        labels=ds.unique_labels,
        sampling_freq_hz=ds.sampling_rate,
        n_bins=25,
    )
    
    # Write the feature dataset to disk with a filename based on the subject ID
    generated_feature_dataset: dsio.Dataset = dsio.Dataset(
        epochs=dataset.x_epochs,
        y_labels=dataset.y_labels,
        subject_id=subject_id,
        stats=dataset.stats,
    )
    dsio.save_dataset_to_file(generated_feature_dataset)
    counter += 1

print(f"\nFeature extraction completed for {counter} subjects.")
print(f"Datasets saved to: {fldr.get_datasets_dir()}")